# 13b — cross-eval: v1-SFT and base on the NEW-dev distribution

**Runtime: A100 (~15 min) or L4 (~30 min).**

Notebook 13 showed SFT v2 losing to v1 on the OLD (docstring-free) dev ruler —
but v2 trained 74% on docstring-style bugs, and the EXAM is docstring-style.
The decision needs the other half of the matrix: how do **v1-SFT** and **base**
score on the same 60 new-dev bugs (k=8) that v2 was measured on?

Full picture after this: {base, SFT v1, SFT v2} × {v0-dev, new-dev} — then we
know whether v2 traded ruler points for exam-style points (good trade) or is
just worse (bad data / bad mixture).

In [ ]:
# Setup — MUST reproduce notebook 13's exact 60-bug new-dev sample (same seed)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
v1_bugs = json.load(open(f'{PHASE8}/data_v1_bugs.json'))
dev_new = [b for b in v1_bugs if b['split'] == 'dev' and b.get('gen') == 'deepseek_v1']
SEED = 3407
dev_new_sample = random.Random(SEED).sample(dev_new, 60)   # identical to notebook 13
print(len(dev_new_sample), 'new-dev bugs (must match notebook 13 sample)')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# Eval machinery (identical settings to notebook 13's new-dev eval: k=8)
import subprocess, tempfile, torch
from concurrent.futures import ThreadPoolExecutor
from unsloth import FastLanguageModel

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

def eval_newdev(model, tok, tag, k=8):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_new_sample:
        text = tok.apply_chat_template(
            [{'role': 'user', 'content': build_repair_prompt(b['buggy'], b['test_list'])}],
            tokenize=False, add_generation_prompt=True)
        enc = tok([text], return_tensors='pt', padding=True, padding_side='left').to('cuda')
        out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                             num_return_sequences=k, max_new_tokens=512,
                             pad_token_id=tok.eos_token_id)
        gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        codes = [extract_code(t) for t in gen]
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    pk = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, f'pass@{k}': pk, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE8}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@8={pk*100:.1f}%")
    return res

# 1) BASE model on new-dev
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1280,
    load_in_4bit=True, dtype=None)
base_res = eval_newdev(model, tok, 'base_newdev')
del model, tok; gc.collect(); torch.cuda.empty_cache()

# 2) v1-SFT (seed 3407, the controlled study's SFT) on new-dev
model, tok = FastLanguageModel.from_pretrained(
    f'{PHASE3}/sft_notrace_s3407_ep1', max_seq_length=1280,
    load_in_4bit=True, dtype=None)
v1_res = eval_newdev(model, tok, 'sftv1_newdev_seed3407')
del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# THE FULL 3x2 MATRIX (v2 rows loaded from notebook 13's saved evals)
def load(tag):
    p = f'{PHASE8}/dev_eval_{tag}.json'
    return json.load(open(p)) if os.path.exists(p) else None
v2e1 = load('sftv2_ep1_newdev_seed3407')
v2e2 = load('sftv2_ep2_newdev_seed3407')
print(f"{'model':<14} {'v0-dev p@1/p@16':<18} {'new-dev p@1/p@8':<18}")
print(f"{'base':<14} {'45.9 / 93.4':<18} {base_res['pass@1']*100:.1f} / {base_res['pass@8']*100:.1f}")
print(f"{'SFT v1':<14} {'58.7 / 91.8':<18} {v1_res['pass@1']*100:.1f} / {v1_res['pass@8']*100:.1f}")
for name, r, v0 in (('SFT v2 ep1', v2e1, '55.1 / 88.5'), ('SFT v2 ep2', v2e2, '56.9 / 83.6')):
    if r:
        print(f"{name:<14} {v0:<18} {r['pass@1']*100:.1f} / {r['pass@8']*100:.1f}")
    else:
        print(f'{name:<14} {v0:<18} (notebook 13 eval json not found on Drive)')
print('\nReading: if v2 >> v1 on new-dev while v1 > v2 on v0-dev, v2 traded old-')
print('format points for exam-format points (the exam is docstring-style) -> the')
print('mixture needs REBALANCING, not abandoning. If v1 >= v2 on BOTH: v2 recipe')
print('is genuinely worse -> investigate mixture/loss before GRPO v2.')

## Bring back to the session
The **3×2 matrix printout** — it decides the v2 mixture question.